# Machine Learning — Hồi quy Stress Level (pipeline nâng cao)

**Pipeline:**
1. Feature mở rộng (11 + feature engineering từ EDA)
2. Baseline + RF/XGB
3. **LightGBM + Optuna** (tune hyperparameter)
4. **CatBoost** (so sánh)
5. **Stacking** (nếu các mô hình chênh RMSE đáng kể)
6. Chọn theo RMSE/MAE (5-fold CV), SHAP, lưu `.pkl`
7. Post-process: clip [1,10], round → 3 mức Low/Medium/High

---
## 1. Import & cấu hình

In [ ]:
import json
import os
import warnings

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')

PROC_PATH = '../data/processed/preprocessed_data.csv'
RAW_PATH = '../data/raw/Teen_Mental_Health_Dataset.csv'
METRICS_PATH = '../results/metrics/ml_metrics.csv'
LGBM_PARAMS_PATH = '../results/metrics/lgbm_best_params.json'
FIG_MODELS_DIR = '../figures/models/'
FIG_SHAP_DIR = '../figures/explainability/'
MODEL_DIR = '../models/ml/'

for d in [os.path.dirname(METRICS_PATH), FIG_MODELS_DIR, FIG_SHAP_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
STRESS_MIN, STRESS_MAX = 1, 10
OPTUNA_TRIALS = 80  # 50–100 trial theo đề xuất
STACKING_MIN_RMSE_GAP = 0.03  # Chênh RMSE giữa model tốt/xấu nhất trong top-3 để bật stacking
CLASS_NAMES = ['Low', 'Medium', 'High']

---
## 2. Dữ liệu & feature set mở rộng

- **Cơ bản (11):** hành vi MXH, ngủ, học tập, …
- **Engineering (4):** từ notebook EDA — `sm_sleep_ratio`, `screen_sleep_ratio`, `heavy_sm_user`, `low_sleep`
- **Loại trừ khỏi X:** `stress_level`, `anxiety_level`, `addiction_level`, `depression_label` (tránh leakage / không phải input hành vi)
- **y:** `stress_level` gốc 1–10 từ file raw

In [ ]:
df_proc = pd.read_csv(PROC_PATH)
df_raw = pd.read_csv(RAW_PATH)
assert len(df_proc) == len(df_raw)

FEATURE_COLS_BASE = [
    'age', 'gender', 'daily_social_media_hours',
    'platform_Both', 'platform_Instagram', 'platform_TikTok',
    'sleep_hours', 'screen_time_before_sleep',
    'academic_performance', 'physical_activity', 'social_interaction_level',
]
FEATURE_COLS_ENG = [
    'sm_sleep_ratio', 'screen_sleep_ratio', 'heavy_sm_user', 'low_sleep',
]
FEATURE_COLS = FEATURE_COLS_BASE + FEATURE_COLS_ENG

missing = [c for c in FEATURE_COLS if c not in df_proc.columns]
if missing:
    raise ValueError(f'Thiếu cột: {missing}')

X = df_proc[FEATURE_COLS].copy()
y = df_raw['stress_level'].astype(float).values
y_mean = float(y.mean())

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'X: {X.shape[0]} mẫu × {X.shape[1]} đặc trưng (11 cơ bản + 4 engineering)')
print(f'y: min={y.min():.0f}, max={y.max():.0f}, mean={y_mean:.2f}')
print(f'Baseline mean predictor — MAE≈{np.mean(np.abs(y - y_mean)):.3f}')

---
## 3. Hàm tiện ích: metrics, post-process, biểu đồ

In [ ]:
def clip_stress(pred: np.ndarray) -> np.ndarray:
    return np.clip(np.asarray(pred, dtype=float), STRESS_MIN, STRESS_MAX)


def round_stress(pred: np.ndarray) -> np.ndarray:
    return np.round(clip_stress(pred)).astype(int)


def stress_to_class(level: float) -> str:
    """Ánh xạ điểm (sau round) sang Low / Medium / High."""
    v = int(round(np.clip(level, STRESS_MIN, STRESS_MAX)))
    if v <= 3:
        return 'Low'
    if v <= 7:
        return 'Medium'
    return 'High'


def compute_metrics(y_true, y_pred) -> dict:
    y_hat = clip_stress(y_pred)
    mse = mean_squared_error(y_true, y_hat)
    return {
        'mae': mean_absolute_error(y_true, y_hat),
        'rmse': float(np.sqrt(mse)),
        'r2': r2_score(y_true, y_hat),
    }


def cv_evaluate(estimator, X_df: pd.DataFrame, y_arr: np.ndarray, kfold):
    """5-fold CV: trả về metrics trung bình + dự đoán OOF."""
    oof = np.zeros(len(y_arr))
    fold_metrics = []
    for train_idx, test_idx in kfold.split(X_df, y_arr):
        model = clone(estimator)
        model.fit(X_df.iloc[train_idx], y_arr[train_idx])
        pred = model.predict(X_df.iloc[test_idx])
        oof[test_idx] = pred
        fold_metrics.append(compute_metrics(y_arr[test_idx], pred))
    df_f = pd.DataFrame(fold_metrics)
    summary = {c: df_f[c].mean() for c in ['mae', 'rmse', 'r2']}
    summary['mae_std'] = df_f['mae'].std()
    summary['rmse_std'] = df_f['rmse'].std()
    summary['r2_std'] = df_f['r2'].std()
    return summary, oof


def plot_pred_vs_actual(y_true, y_pred, title: str, save_path: str):
    y_hat = clip_stress(y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_hat, alpha=0.45, edgecolors='none')
    ax.plot([STRESS_MIN, STRESS_MAX], [STRESS_MIN, STRESS_MAX], 'r--', lw=2, label='y=x')
    ax.set_xlim(STRESS_MIN - 0.5, STRESS_MAX + 0.5)
    ax.set_ylim(STRESS_MIN - 0.5, STRESS_MAX + 0.5)
    ax.set_xlabel('Stress thực tế')
    ax.set_ylabel('Stress dự đoán (clip)')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_residuals(y_true, y_pred, title: str, save_path: str):
    residuals = y_true - clip_stress(y_pred)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.histplot(residuals, bins=25, kde=True, ax=ax)
    ax.axvline(0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Residual')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

---
## 4. Baseline & mô hình cổ điển (RF, XGB)

In [ ]:
baseline_models = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'XGBoost': xgb.XGBRegressor(
        objective='reg:squarederror', n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

results = []
oof_predictions = {}

for name, est in baseline_models.items():
    print(f'CV — {name}...')
    summary, oof = cv_evaluate(est, X, y, kf)
    oof_predictions[name] = oof
    results.append({'model': name, 'task': 'regression', 'tuned': False, **summary})
    print(f"  MAE={summary['mae']:.3f}  RMSE={summary['rmse']:.3f}  R²={summary['r2']:.3f}")

---
## 5. LightGBM + Optuna (80 trials)

Tối ưu **RMSE** trên 5-fold CV (trung bình).

In [ ]:
def lgb_objective(trial: optuna.Trial) -> float:
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': RANDOM_STATE,
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    rmses = []
    for train_idx, test_idx in kf.split(X, y):
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X.iloc[train_idx], y[train_idx],
            eval_set=[(X.iloc[test_idx], y[test_idx])],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        pred = model.predict(X.iloc[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], clip_stress(pred))))
    return float(np.mean(rmses))


study = optuna.create_study(direction='minimize', study_name='lgbm_stress')
study.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

lgbm_best_params = study.best_params
lgbm_best_params.update({'objective': 'regression', 'random_state': RANDOM_STATE, 'verbosity': -1})

with open(LGBM_PARAMS_PATH, 'w', encoding='utf-8') as f:
    json.dump(lgbm_best_params, f, indent=2)

print(f'Best Optuna RMSE: {study.best_value:.4f}')
print(f'Best params saved: {LGBM_PARAMS_PATH}')
display(pd.Series(lgbm_best_params))

In [ ]:
lgbm_tuned = lgb.LGBMRegressor(**lgbm_best_params)
print('CV — LightGBM (tuned)...')
lgbm_summary, lgbm_oof = cv_evaluate(lgbm_tuned, X, y, kf)
oof_predictions['LightGBM (tuned)'] = lgbm_oof
results.append({'model': 'LightGBM (tuned)', 'task': 'regression', 'tuned': True, **lgbm_summary})
print(f"  MAE={lgbm_summary['mae']:.3f}  RMSE={lgbm_summary['rmse']:.3f}  R²={lgbm_summary['r2']:.3f}")

---
## 6. CatBoost Regressor (so sánh)

In [ ]:
catboost_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False,
)

print('CV — CatBoost...')
cat_summary, cat_oof = cv_evaluate(catboost_model, X, y, kf)
oof_predictions['CatBoost'] = cat_oof
results.append({'model': 'CatBoost', 'task': 'regression', 'tuned': False, **cat_summary})
print(f"  MAE={cat_summary['mae']:.3f}  RMSE={cat_summary['rmse']:.3f}  R²={cat_summary['r2']:.3f}")

---
## 7. Stacking (tuỳ chọn)

Kích hoạt khi RMSE của các mô hình **LightGBM, CatBoost, XGBoost** chênh ≥ `STACKING_MIN_RMSE_GAP`.

In [ ]:
df_tmp = pd.DataFrame(results).set_index('model')
stack_candidates = ['LightGBM (tuned)', 'CatBoost', 'XGBoost']
use_stacking = False
stack_rmse = df_tmp.loc[stack_candidates, 'rmse'].astype(float)
rmse_gap = stack_rmse.max() - stack_rmse.min()

print('RMSE các ứng viên stacking:')
print(stack_rmse)
print(f'Chênh lệch max-min: {rmse_gap:.4f} (ngưỡng {STACKING_MIN_RMSE_GAP})')

use_stacking = rmse_gap >= STACKING_MIN_RMSE_GAP

if use_stacking:
    print('\n→ Bật StackingRegressor (Ridge meta-learner)')
    stacking_est = StackingRegressor(
        estimators=[
            ('lgbm', clone(lgbm_tuned)),
            ('cat', clone(catboost_model)),
            ('xgb', clone(baseline_models['XGBoost'])),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=-1,
    )
    print('CV — Stacking...')
    stack_summary, stack_oof = cv_evaluate(stacking_est, X, y, kf)
    oof_predictions['Stacking (LGBM+Cat+XGB)'] = stack_oof
    results.append({
        'model': 'Stacking (LGBM+Cat+XGB)',
        'task': 'regression',
        'tuned': True,
        **stack_summary,
    })
    print(f"  MAE={stack_summary['mae']:.3f}  RMSE={stack_summary['rmse']:.3f}  R²={stack_summary['r2']:.3f}")
else:
    print('\n→ Bỏ qua stacking (các mô hình gần nhau, không đáng kể)')

---
## 8. Bảng so sánh & chọn mô hình tốt nhất (RMSE)

In [ ]:
df_metrics = pd.DataFrame(results)
df_metrics = df_metrics.sort_values('rmse', ascending=True).reset_index(drop=True)

display_cols = [
    'model', 'task', 'tuned', 'mae', 'rmse', 'r2',
    'mae_std', 'rmse_std', 'r2_std',
]
print('=== Bảng so sánh (5-fold CV, feature mở rộng) ===')
display(df_metrics[display_cols])

df_metrics[display_cols].to_csv(METRICS_PATH, index=False)
print(f'\n Đã lưu: {METRICS_PATH}')

best_model_name = df_metrics.loc[0, 'model']
best_rmse = df_metrics.loc[0, 'rmse']
best_mae = df_metrics.loc[0, 'mae']
print(f'\nMô hình tốt nhất: {best_model_name} | RMSE={best_rmse:.3f} | MAE={best_mae:.3f}')

---
## 9. Biểu đồ OOF (mô hình thắng) + post-process 3 mức

In [ ]:
best_oof = oof_predictions[best_model_name]
best_oof_clip = clip_stress(best_oof)
best_oof_round = round_stress(best_oof)
best_oof_class = np.array([stress_to_class(v) for v in best_oof_round])

m_cont = compute_metrics(y, best_oof)
m_round = compute_metrics(y, best_oof_round)
print(f'OOF liên tục (clip):  MAE={m_cont["mae"]:.3f}  RMSE={m_cont["rmse"]:.3f}  R²={m_cont["r2"]:.3f}')
print(f'OOF sau round:       MAE={m_round["mae"]:.3f}  RMSE={m_round["rmse"]:.3f}  R²={m_round["r2"]:.3f}')
print('\nPhân phối lớp dự đoán (round + 3 mức):')
print(pd.Series(best_oof_class).value_counts().reindex(CLASS_NAMES))

slug = best_model_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
plot_pred_vs_actual(
    y, best_oof,
    title=f'Predicted vs Actual (OOF) — {best_model_name}',
    save_path=os.path.join(FIG_MODELS_DIR, f'pred_vs_actual_{slug}.png'),
)
plot_residuals(
    y, best_oof,
    title=f'Residuals (OOF) — {best_model_name}',
    save_path=os.path.join(FIG_MODELS_DIR, f'residuals_{slug}.png'),
)

---
## 10. Huấn luyện cuối, SHAP & lưu mô hình

In [ ]:
MODEL_BUILDERS = {
    'Dummy (mean)': lambda: DummyRegressor(strategy='mean'),
    'Random Forest': lambda: clone(baseline_models['Random Forest']),
    'XGBoost': lambda: clone(baseline_models['XGBoost']),
    'LightGBM (tuned)': lambda: lgb.LGBMRegressor(**lgbm_best_params),
    'CatBoost': lambda: clone(catboost_model),
    'Stacking (LGBM+Cat+XGB)': lambda: StackingRegressor(
        estimators=[
            ('lgbm', lgb.LGBMRegressor(**lgbm_best_params)),
            ('cat', clone(catboost_model)),
            ('xgb', clone(baseline_models['XGBoost'])),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=-1,
    ),
}

if best_model_name not in MODEL_BUILDERS:
    raise KeyError(f'Chưa định nghĩa builder cho: {best_model_name}')

best_estimator = MODEL_BUILDERS[best_model_name]()
best_estimator.fit(X, y)

model_slug = slug
model_path = os.path.join(MODEL_DIR, f'best_stress_regressor_{model_slug}.pkl')

artifact = {
    'model': best_estimator,
    'task': 'regression',
    'target': 'stress_level',
    'target_range': [STRESS_MIN, STRESS_MAX],
    'feature_cols': FEATURE_COLS,
    'best_model_name': best_model_name,
    'cv_rmse': float(best_rmse),
    'cv_mae': float(best_mae),
    'cv_r2': float(df_metrics.loc[0, 'r2']),
    'lgbm_best_params': lgbm_best_params if 'LightGBM' in best_model_name or use_stacking else None,
    'postprocess': {
        'clip': [STRESS_MIN, STRESS_MAX],
        'round': True,
        'class_bins': {'Low': '1-3', 'Medium': '4-7', 'High': '8-10'},
    },
}
joblib.dump(artifact, model_path)
print(f'Đã lưu mô hình: {model_path}')

In [ ]:
# SHAP trên mô hình cây đơn (Stacking → dùng LightGBM tuned để giải thích)
shap_model = best_estimator
shap_model_name = best_model_name
if 'Stacking' in best_model_name:
    shap_model = lgb.LGBMRegressor(**lgbm_best_params)
    shap_model.fit(X, y)
    shap_model_name = 'LightGBM (tuned) — proxy cho SHAP'

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, feature_names=FEATURE_COLS, show=False)
plt.title(f'SHAP — {shap_model_name}')
plt.tight_layout()
shap_path = os.path.join(FIG_SHAP_DIR, 'shap_importance.png')
plt.savefig(shap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f' Đã lưu SHAP: {shap_path}')

---
## 11. Hàm dự đoán (deploy / demo)

```python
def predict_stress(model, X_row, clip=True, to_class=False):
    raw = model.predict(X_row)
    if clip:
        raw = np.clip(raw, 1, 10)
    if to_class:
        return stress_to_class(round(raw[0]))
    return raw
```

In [ ]:
def predict_stress(model, X_row: pd.DataFrame, to_class: bool = False):
    """Dự đoán 1 hoặc nhiều mẫu; tuỳ chọn trả về nhãn 3 mức."""
    raw = clip_stress(model.predict(X_row[FEATURE_COLS]))
    if to_class:
        return [stress_to_class(v) for v in round_stress(raw)]
    return raw


# Demo 3 mẫu đầu
demo = predict_stress(best_estimator, X.head(3), to_class=True)
for i, (yt, yp, cls) in enumerate(zip(y[:3], round_stress(best_estimator.predict(X.head(3))), demo)):
    print(f'Mẫu {i}: thực tế={yt:.0f} | dự đoán={yp} | lớp={cls}')

---
## 12. Tóm tắt đầu ra

| File | Nội dung |
|------|----------|
| `results/metrics/ml_metrics.csv` | So sánh tất cả mô hình |
| `results/metrics/lgbm_best_params.json` | Hyperparameter Optuna |
| `figures/models/pred_vs_actual_*.png` | OOF mô hình thắng |
| `figures/explainability/shap_importance.png` | SHAP |
| `models/ml/best_stress_regressor_*.pkl` | Artifact + metadata |